# Nano PSM Retention Blend From Reviewed Checkpoint

Train a recovery checkpoint on reviewed, incremental, and retention data, initialized from the reviewed-5k best checkpoint.

In [ ]:
!pip install -U huggingface_hub datasets safetensors onnx onnxruntime
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

In [ ]:
from huggingface_hub import login
login()

In [ ]:
HF_DATASET_REPO = 'chkrishna2001/nano-psm-retention-blend-7k'
HF_PREVIOUS_CHECKPOINT_REPO = 'chkrishna2001/nano-psm-primary-10m-reviewed-5k-checkpoints'
HF_OUTPUT_CHECKPOINT_REPO = 'chkrishna2001/nano-psm-primary-10m-retention-blend-from-reviewed-checkpoints'
CODE_REPO = 'https://github.com/chkrishna2001/PSM.git'
CODE_REVISION = 'main'

LOCAL_DATA_DIR = '/content/psm-retention-blend-7k'
LOCAL_PREVIOUS_CHECKPOINT_DIR = '/content/nano-psm-primary-10m-reviewed-5k-checkpoints'
LOCAL_OUTPUT_CHECKPOINT_DIR = '/content/nano-psm-primary-10m-retention-blend-from-reviewed-checkpoints'
REPO_DIR = '/content/PSM'


## Download Dataset, Code, And Reviewed Checkpoint

In [ ]:
from huggingface_hub import snapshot_download

dataset_dir = snapshot_download(repo_id=HF_DATASET_REPO, repo_type='dataset', local_dir=LOCAL_DATA_DIR, force_download=True)
print('dataset:', dataset_dir)

previous_checkpoint_dir = snapshot_download(
    repo_id=HF_PREVIOUS_CHECKPOINT_REPO,
    repo_type='model',
    local_dir=LOCAL_PREVIOUS_CHECKPOINT_DIR,
    force_download=True,
)
print('reviewed checkpoint:', previous_checkpoint_dir)

try:
    output_checkpoint_dir = snapshot_download(
        repo_id=HF_OUTPUT_CHECKPOINT_REPO,
        repo_type='model',
        local_dir=LOCAL_OUTPUT_CHECKPOINT_DIR,
        force_download=True,
    )
    print('resume checkpoint:', output_checkpoint_dir)
except Exception as exc:
    output_checkpoint_dir = LOCAL_OUTPUT_CHECKPOINT_DIR
    print('No blend checkpoint snapshot yet; will initialize from reviewed checkpoint:', exc)

!git clone {CODE_REPO} {REPO_DIR} || true
%cd {REPO_DIR}
!git fetch origin
!git checkout {CODE_REVISION}
!git pull --ff-only || true
!git rev-parse --short HEAD

## Preflight

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

train_path = Path(LOCAL_DATA_DIR) / 'train.jsonl'
validation_path = Path(LOCAL_DATA_DIR) / 'validation.jsonl'
metadata_path = Path(LOCAL_DATA_DIR) / 'metadata.json'
config_path = Path('nano-psm/configs/primary-10m.json')
init_checkpoint_path = Path(LOCAL_PREVIOUS_CHECKPOINT_DIR) / 'checkpoint-best.pt'
for required in (train_path, validation_path, metadata_path, config_path, init_checkpoint_path):
    assert required.exists(), f'Missing required file: {required}'

sys.path.insert(0, 'nano-psm/src')
from nano_psm.model import NanoPsmConfig
from nano_psm.dataset import load_jsonl

metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
config_doc = json.loads(config_path.read_text(encoding='utf-8'))
model_config = NanoPsmConfig(**config_doc['model'])
train_examples = load_jsonl(train_path)
validation_examples = load_jsonl(validation_path)

print('dataset', HF_DATASET_REPO)
print('previous_checkpoint', HF_PREVIOUS_CHECKPOINT_REPO)
print('output_checkpoint', HF_OUTPUT_CHECKPOINT_REPO)
print('metadata_total_examples', metadata.get('total_examples'))
print('train_examples', len(train_examples))
print('validation_examples', len(validation_examples))
print('action_mix', metadata.get('action_mix'))
print('source_mix', metadata.get('source_mix'))
print('parameter_estimate', f'{model_config.estimate_parameters():,}')
assert len(train_examples) == 7000
assert len(validation_examples) == 700
assert metadata.get('total_examples') == 7700
assert 8_000_000 <= model_config.estimate_parameters() <= 12_000_000

subprocess.run([sys.executable, '-m', 'compileall', 'nano-psm/src/nano_psm'], check=True)
print('preflight ok')

## Train Blend Curriculum

In [ ]:
MAX_STEPS = 3000
EVAL_EVERY = 250
SAVE_EVERY = 500

!python nano-psm/src/nano_psm/train.py \
  --config nano-psm/configs/primary-10m.json \
  --train {LOCAL_DATA_DIR}/train.jsonl \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint-dir {LOCAL_OUTPUT_CHECKPOINT_DIR} \
  --init-checkpoint {LOCAL_PREVIOUS_CHECKPOINT_DIR}/checkpoint-best.pt \
  --resume auto \
  --device auto \
  --max-steps {MAX_STEPS} \
  --eval-every {EVAL_EVERY} \
  --save-every {SAVE_EVERY}

## Evaluate Blend Validation

In [ ]:
!python nano-psm/src/nano_psm/evaluate.py \
  --config nano-psm/configs/primary-10m.json \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint {LOCAL_OUTPUT_CHECKPOINT_DIR}/checkpoint-best.pt \
  --device auto

## Cross-Evaluate Current Checkpoint

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

CROSS_EVAL_DATASETS = {
    'reviewed_5k': ('chkrishna2001/nano-psm-fast-mixed-reviewed-5k', '/content/psm-fast-mixed-reviewed-5k'),
    'reviewed_v2': ('chkrishna2001/nano-psm-fast-mixed-reviewed-v2-5k', '/content/psm-fast-mixed-reviewed-v2-5k'),
    'incremental_5k': ('chkrishna2001/nano-psm-fast-mixed-reviewed-incremental-5k', '/content/psm-fast-mixed-reviewed-incremental-5k'),
    'retention_decay_5k': ('chkrishna2001/nano-psm-retention-decay-5k', '/content/psm-retention-decay-5k'),
    'retention_blend_7k': (HF_DATASET_REPO, LOCAL_DATA_DIR),
}

for name, (repo_id, local_dir) in CROSS_EVAL_DATASETS.items():
    dataset_path = snapshot_download(repo_id=repo_id, repo_type='dataset', local_dir=local_dir, force_download=True)
    validation_path = Path(dataset_path) / 'validation.jsonl'
    assert validation_path.exists(), f'Missing validation file for {name}: {validation_path}'
    print(name, validation_path)


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

cross_eval_results = {}
checkpoint_path = f'{LOCAL_OUTPUT_CHECKPOINT_DIR}/checkpoint-best.pt'
for name, (_, local_dir) in CROSS_EVAL_DATASETS.items():
    validation_path = f'{local_dir}/validation.jsonl'
    cmd = [
        sys.executable,
        'nano-psm/src/nano_psm/evaluate.py',
        '--config', 'nano-psm/configs/primary-10m.json',
        '--validation', validation_path,
        '--checkpoint', checkpoint_path,
        '--device', 'auto',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    metrics = json.loads(result.stdout)
    cross_eval_results[name] = metrics
    print('\n' + name)
    print(json.dumps(metrics, indent=2))

out = Path('/content/cross-eval-retention-blend-3000.json')
out.write_text(json.dumps(cross_eval_results, indent=2), encoding='utf-8')
print('\nwritten:', out, 'bytes:', out.stat().st_size)


In [ ]:
from google.colab import files
files.download('/content/cross-eval-retention-blend-3000.json')

## Export Blend Predictions

In [ ]:
import subprocess, sys, json
from pathlib import Path
from google.colab import files

out = Path('/content/validationresults-retention-blend-3000-show-correct.json')
cmd = [
    sys.executable,
    'nano-psm/src/nano_psm/inspect_predictions.py',
    '--config', 'nano-psm/configs/primary-10m.json',
    '--validation', f'{LOCAL_DATA_DIR}/validation.jsonl',
    '--checkpoint', f'{LOCAL_OUTPUT_CHECKPOINT_DIR}/checkpoint-best.pt',
    '--device', 'auto',
    '--limit', '100000',
    '--show-correct',
]
result = subprocess.run(cmd, capture_output=True, text=True, check=True)
out.write_text(result.stdout, encoding='utf-8')
rows = json.loads(result.stdout)
print('rows:', len(rows))
print('bytes:', out.stat().st_size)
files.download(str(out))

## Upload Blend Checkpoints And Validation Results

In [ ]:
from huggingface_hub import create_repo, upload_file, upload_folder
from pathlib import Path

create_repo(repo_id=HF_OUTPUT_CHECKPOINT_REPO, repo_type='model', private=True, exist_ok=True)
upload_folder(
    repo_id=HF_OUTPUT_CHECKPOINT_REPO,
    repo_type='model',
    folder_path=LOCAL_OUTPUT_CHECKPOINT_DIR,
    path_in_repo='.',
    commit_message='sync nano psm retention blend checkpoint from reviewed base'
)

ARTIFACT_UPLOADS = {
    '/content/cross-eval-retention-blend-3000.json': 'validation/cross-eval-retention-blend-3000.json',
    '/content/validationresults-retention-blend-3000-show-correct.json': 'validation/validationresults-retention-blend-3000-show-correct.json',
}

for local_path, path_in_repo in ARTIFACT_UPLOADS.items():
    path = Path(local_path)
    if not path.exists():
        print('skip missing validation artifact:', path)
        continue
    upload_file(
        repo_id=HF_OUTPUT_CHECKPOINT_REPO,
        repo_type='model',
        path_or_fileobj=str(path),
        path_in_repo=path_in_repo,
        commit_message=f'upload {path.name}'
    )
    print('uploaded validation artifact:', path, '->', path_in_repo)